In [ ]:
# Importing the required libraries
import os
import torch
import cv2
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
)
import torchvision.models as models
import albumentations as A
from albumentations import Compose, HorizontalFlip, VerticalFlip, Rotate, MotionBlur, GaussianBlur, Resize
from tqdm import tqdm
import pandas as pd
import copy



In [ ]:
# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
NUM_EPOCHS = 10
KFOLD_SPLITS = 10
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/Mango_Fruit_Dataset

Anthracnose  Bacterial_Black_Spot  Healthy  Multiple


In [ ]:
# Refering to the Fruit dataset stored in my Google Drive (if can varry for you)
fruit_input = "/content/drive/MyDrive/Mango_Fruit_Dataset"

In [ ]:
# Novel Class-wise fruit-specific augmentations
class_specific_fruit_transforms = {
    "Anthracnose": A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.MotionBlur(p=0.4),
        A.Rotate(limit=15, p=0.5)
    ]),
    "Bacterial_Black_Spot": A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.VerticalFlip(p=0.5),
        A.GaussianBlur(p=0.4),
        A.Rotate(limit=10, p=0.4)
    ]),
    "Healthy": A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.3),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=5, p=0.3)
    ]),
    "Multiple": A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=20, p=0.5)
    ])
}

#  Validation/Test transform (same as leaf)
val_test_transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])

#  Wrapper
def wrap_transform(transform):
    return lambda image: transform(image=image)["image"]

#  Wrapped versions for fruit
wrapped_class_specific_fruit_transforms = {
    cls: wrap_transform(tfm) for cls, tfm in class_specific_fruit_transforms.items()
}

wrapped_val_test_transform = wrap_transform(val_test_transform)


In [ ]:

class LeafDataset(Dataset):
    def __init__(self, root_dir, transform_dict=None):
        self.root_dir = root_dir
        self.transform_dict = transform_dict
        self.image_paths = []
        self.labels = []
        self.class_names = sorted(os.listdir(root_dir))

        for label_idx, class_name in enumerate(self.class_names):
            class_folder = os.path.join(root_dir, class_name)
            for img_file in os.listdir(class_folder):
                self.image_paths.append(os.path.join(class_folder, img_file))
                self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        class_name = self.class_names[label]

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform_dict and class_name in self.transform_dict:
            image = self.transform_dict[class_name](image)
        else:
            image = A.Resize(224, 224)(image=image)["image"]

        image = torch.tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)
        return image, label


FruitDataset = LeafDataset

In [ ]:


def get_model(model_name="resnet18", num_classes=4, pretrained=True):
    if model_name == "resnet18":
        model = models.resnet18(pretrained=pretrained)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "resnet50":
        model = models.resnet50(pretrained=pretrained)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(pretrained=pretrained)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=pretrained)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_tiny":
      model= models.convnext_tiny(pretrained=pretrained)
      in_features = model.classifier[2].in_features
      model.classifier[2] = nn.Linear(in_features,num_classes)

    else:
        raise ValueError("Unsupported model name!")

    return model


In [ ]:
fruit_dataset = FruitDataset(
    root_dir=fruit_input,
    transform_dict=wrapped_class_specific_fruit_transforms
)

In [ ]:
def run_kfold_fruit_training(
    model_name,
    fruit_dataset,
    transform_dict_train,
    transform_val,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=5,
    save_prefix="fruit_model",
    lr=1e-4,
    seed=42,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
):
    from torch.utils.data import Subset, DataLoader
    from sklearn.model_selection import KFold
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
    )
    import pandas as pd
    import copy
    import torch.nn as nn
    import torch.optim as optim
    from tqdm import tqdm
    import torch

    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=seed)
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(fruit_dataset)):
        print(f"\n🟩 Fold {fold + 1}/{k_folds}")

        train_subset = Subset(fruit_dataset, train_idx)
        val_subset = Subset(fruit_dataset, val_idx)

        train_subset.dataset.transform_dict = transform_dict_train
        val_subset.dataset.transform_dict = {
            cls: transform_val for cls in fruit_dataset.class_names
        }

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

        model = get_model(model_name, num_classes=num_classes).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
        criterion = nn.CrossEntropyLoss()

        best_val_acc = 0
        best_model_wts = copy.deepcopy(model.state_dict())

        for epoch in range(num_epochs):
            print(f"\n📚 Epoch {epoch+1}")
            model.train()
            train_loss = 0

            for images, labels in tqdm(train_loader):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            scheduler.step()

            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    preds = torch.argmax(outputs, dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            acc = accuracy_score(all_labels, all_preds)
            prec = precision_score(all_labels, all_preds, average='macro')
            rec = recall_score(all_labels, all_preds, average='macro')
            f1 = f1_score(all_labels, all_preds, average='macro')
            ber = 1 - balanced_accuracy_score(all_labels, all_preds)
            kappa = cohen_kappa_score(all_labels, all_preds)
            mcc = matthews_corrcoef(all_labels, all_preds)

            print(f"✅ Fold {fold+1} Epoch {epoch+1} — Acc: {acc:.4f}, F1: {f1:.4f}, BER: {ber:.4f}")

            if acc > best_val_acc:
                best_val_acc = acc
                best_model_wts = copy.deepcopy(model.state_dict())

        torch.save(best_model_wts, f"{save_prefix}_fold{fold+1}.pth")

        fold_metrics.append({
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "BER": ber,
            "Kappa": kappa,
            "MCC": mcc
        })

    df_metrics = pd.DataFrame(fold_metrics)

    print("\n📈 K-Fold Results Summary:")
    print(df_metrics)

    print(f"\n📊 Mean ± Std across folds for {save_prefix}:")
    for metric in ["Accuracy", "Precision", "Recall", "F1 Score", "BER", "Kappa", "MCC"]:
        mean = df_metrics[metric].mean()
        std = df_metrics[metric].std()
        print(f"{metric}: {mean:.4f} ± {std:.4f}")

    return df_metrics


In [ ]:


df_resnet_fruit = run_kfold_fruit_training(
    model_name="resnet50",
    fruit_dataset=fruit_dataset,
    transform_dict_train=wrapped_class_specific_fruit_transforms,
    transform_val=wrapped_val_test_transform,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=10,
    save_prefix="resnet50_fruit"
)



🟩 Fold 1/10


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 207MB/s]



📚 Epoch 1


100%|██████████| 53/53 [15:15<00:00, 17.27s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9787, F1: 0.9772, BER: 0.0207

📚 Epoch 2


100%|██████████| 53/53 [00:26<00:00,  2.03it/s]


... (211 outputs left out)

100%|██████████| 53/53 [00:26<00:00,  2.01it/s]


✅ Fold 10 Epoch 9 — Acc: 0.9786, F1: 0.9671, BER: 0.0417

📚 Epoch 10


100%|██████████| 53/53 [00:26<00:00,  2.01it/s]


✅ Fold 10 Epoch 10 — Acc: 0.9786, F1: 0.9671, BER: 0.0417

📈 K-Fold Results Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
1     2  0.994652   0.996377  0.989130  0.992620  0.010870  0.992146  0.992190
2     3  0.989305   0.979096  0.979096  0.979096  0.020904  0.984067  0.984067
3     4  0.983957   0.977635  0.983165  0.979942  0.016835  0.977140  0.977261
4     5  0.983957   0.976587  0.974024  0.975128  0.025976  0.977115  0.977155
5     6  0.978610   0.981685  0.977609  0.979435  0.022391  0.970353  0.970508
6     7  0.978610   0.980263  0.952381  0.963594  0.047619  0.969473  0.970001
7     8  0.994652   0.988095  0.995455  0.991609  0.004545  0.992438  0.992479
8     9  0.973262   0.955414  0.976331  0.965242  0.023669  0.960787  0.961096
9    10  0.978610   0.980125  0.958333  0.967114  0.041667  0.969729  0.970250

📊 Mean ± Std across folds for resnet50_fruit:

In [ ]:


df_mobileNet_fruit = run_kfold_fruit_training(
    model_name="mobilenet_v2",
    fruit_dataset=fruit_dataset,
    transform_dict_train=wrapped_class_specific_fruit_transforms,
    transform_val=wrapped_val_test_transform,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=10,
    save_prefix="mobilenet_v2"
)



🟩 Fold 1/10


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 155MB/s]



📚 Epoch 1


100%|██████████| 53/53 [00:16<00:00,  3.18it/s]


✅ Fold 1 Epoch 1 — Acc: 0.9894, F1: 0.9874, BER: 0.0166

📚 Epoch 2


100%|██████████| 53/53 [00:16<00:00,  3.28it/s]


... (195 outputs left out)

100%|██████████| 53/53 [00:16<00:00,  3.23it/s]


✅ Fold 10 Epoch 9 — Acc: 0.9840, F1: 0.9767, BER: 0.0312

📚 Epoch 10


100%|██████████| 53/53 [00:16<00:00,  3.22it/s]


✅ Fold 10 Epoch 10 — Acc: 0.9840, F1: 0.9767, BER: 0.0312

📈 K-Fold Results Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  0.994681   0.987500  0.996575  0.991866  0.003425  0.992423  0.992465
1     2  0.989305   0.992857  0.978261  0.985013  0.021739  0.984263  0.984437
2     3  0.983957   0.964440  0.974859  0.969396  0.025141  0.976146  0.976189
3     4  0.994652   0.990385  0.996212  0.993190  0.003788  0.992379  0.992421
4     5  0.983957   0.977008  0.980656  0.978752  0.019344  0.977149  0.977189
5     6  0.978610   0.975525  0.977609  0.976300  0.022391  0.970396  0.970511
6     7  0.978610   0.978251  0.952381  0.962560  0.047619  0.969494  0.969979
7     8  0.994652   0.995536  0.987500  0.991337  0.012500  0.992417  0.992458
8     9  0.978610   0.965332  0.979272  0.972060  0.020728  0.968545  0.968679
9    10  0.983957   0.987069  0.968750  0.976696  0.031250  0.977307  0.977671

📊 Mean ± Std across folds for mobilenet_v2:
A

In [ ]:


df_efficientNet_fruit = run_kfold_fruit_training(
    model_name="efficientnet_b0",
    fruit_dataset=fruit_dataset,
    transform_dict_train=wrapped_class_specific_fruit_transforms,
    transform_val=wrapped_val_test_transform,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=10,
    save_prefix="efficientnet_b0"
)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth



🟩 Fold 1/10


100%|██████████| 20.5M/20.5M [00:00<00:00, 180MB/s]



📚 Epoch 1


100%|██████████| 53/53 [00:18<00:00,  2.91it/s]


✅ Fold 1 Epoch 1 — Acc: 0.9468, F1: 0.9278, BER: 0.0859

📚 Epoch 2


... (200 outputs left out)

100%|██████████| 53/53 [00:18<00:00,  2.89it/s]


✅ Fold 10 Epoch 9 — Acc: 0.9893, F1: 0.9847, BER: 0.0208

📚 Epoch 10


100%|██████████| 53/53 [00:18<00:00,  2.93it/s]


✅ Fold 10 Epoch 10 — Acc: 0.9840, F1: 0.9773, BER: 0.0254

📈 K-Fold Results Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  0.994681   0.987500  0.996575  0.991866  0.003425  0.992423  0.992465
1     2  0.994652   0.996377  0.989130  0.992620  0.010870  0.992146  0.992190
2     3  0.989305   0.970588  0.992751  0.980723  0.007249  0.984143  0.984279
3     4  0.994652   0.990385  0.996212  0.993190  0.003788  0.992379  0.992421
4     5  0.989305   0.984893  0.984893  0.984893  0.015107  0.984748  0.984748
5     6  0.973262   0.971516  0.971511  0.971011  0.028489  0.962966  0.963234
6     7  0.983957   0.984492  0.964286  0.972890  0.035714  0.977136  0.977421
7     8  0.994652   0.988095  0.995455  0.991609  0.004545  0.992438  0.992479
8     9  0.973262   0.955414  0.976331  0.965242  0.023669  0.960787  0.961096
9    10  0.983957   0.980202  0.974621  0.977286  0.025379  0.977364  0.977404

📊 Mean ± Std across folds for efficientnet_b0

In [ ]:
df_convNext_fruit = run_kfold_fruit_training(
    model_name="convnext_tiny",
    fruit_dataset=fruit_dataset,
    transform_dict_train=wrapped_class_specific_fruit_transforms,
    transform_val=wrapped_val_test_transform,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=10,
    save_prefix="convnext_tiny"
)



🟩 Fold 1/10


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1`. You can also use `weights=ConvNeXt_Tiny_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:00<00:00, 186MB/s]



📚 Epoch 1


100%|██████████| 53/53 [11:55<00:00, 13.50s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9840, F1: 0.9725, BER: 0.0395

📚 Epoch 2


100%|██████████| 53/53 [00:34<00:00,  1.54it/s]


... (211 outputs left out)

100%|██████████| 53/53 [00:35<00:00,  1.47it/s]


✅ Fold 10 Epoch 9 — Acc: 0.9840, F1: 0.9767, BER: 0.0312

📚 Epoch 10


100%|██████████| 53/53 [00:36<00:00,  1.47it/s]


✅ Fold 10 Epoch 10 — Acc: 0.9840, F1: 0.9767, BER: 0.0312

📈 K-Fold Results Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  0.989362   0.991935  0.973684  0.982013  0.026316  0.984763  0.984928
1     2  0.983957   0.981390  0.974584  0.977859  0.025416  0.976439  0.976482
2     3  0.983957   0.958333  0.988513  0.971447  0.011487  0.976259  0.976568
3     4  0.994652   0.990385  0.996212  0.993190  0.003788  0.992379  0.992421
4     5  0.989305   0.988021  0.978261  0.982567  0.021739  0.984721  0.984843
5     6  0.978610   0.981685  0.977609  0.979435  0.022391  0.970353  0.970508
6     7  0.978610   0.980263  0.952381  0.963594  0.047619  0.969473  0.970001
7     8  0.994652   0.988095  0.995455  0.991609  0.004545  0.992438  0.992479
8     9  0.983957   0.976190  0.982213  0.979112  0.017787  0.976345  0.976389
9    10  0.983957   0.987069  0.968750  0.976696  0.031250  0.977307  0.977671

📊 Mean ± Std across folds for convnext_tiny:
